# Caseware Customer Voice — NLP Sentiment & Pain Point Analysis

**Full pipeline: Data Parsing → Sentiment Analysis (VADER) → Pain Point Categorization → Topic Modeling (TF-IDF + NMF)**

Built by Hammad Mirza | April 2026

---

This notebook walks through the complete NLP pipeline for analyzing 221 customer reviews of Caseware products from G2, Capterra, Software Advice, and Reddit.

## Step 0: Install Dependencies

In [ ]:
!pip install vaderSentiment scikit-learn pandas plotly -q

In [ ]:
import pandas as pd
import numpy as np
import re
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import plotly.graph_objects as go
import plotly.express as px
from collections import Counter

print('All imports loaded successfully.')

---
## Step 1: Load the Data

The raw text files from G2, Capterra, Software Advice, and Reddit were parsed using `parse_reviews.py` into a structured CSV. Each row has:
- **source**: which platform the review came from
- **rating**: star rating (0 if not available, e.g. Reddit)
- **text**: the full review text (pros + cons combined)
- **pros**: the positive comments (where available)
- **cons**: the negative comments (where available)

Reddit comments were filtered to only include those mentioning Caseware-related keywords (caseware, working papers, DAS, trial balance, cloud version, etc.), reducing 279 raw comments to 94 relevant ones.

In [ ]:
# Load the parsed reviews
# If running in Colab, upload reviews.csv first or mount Google Drive
df = pd.read_csv('reviews.csv')

# Clean up
df['text'] = df['text'].fillna('').astype(str)
df['pros'] = df['pros'].fillna('').astype(str)
df['cons'] = df['cons'].fillna('').astype(str)

# Remove entries with very short text (likely parsing artifacts)
df = df[df['text'].str.len() > 20].reset_index(drop=True)

print(f'Total reviews loaded: {len(df)}')
print(f'\nReviews by source:')
print(df['source'].value_counts())
print(f'\nReviews with star ratings: {len(df[df.rating > 0])}')
print(f'Average star rating: {df[df.rating > 0]["rating"].mean():.2f}')

In [ ]:
# Preview the data
print('=== SAMPLE REVIEWS ===')
for source in df['source'].unique():
    sample = df[df['source'] == source].iloc[0]
    print(f'\n--- {source} ---')
    print(f'Rating: {sample["rating"]}')
    print(f'Text: {sample["text"][:200]}...')
    if sample['pros']:
        print(f'Pros: {sample["pros"][:100]}...')
    if sample['cons']:
        print(f'Cons: {sample["cons"][:100]}...')

---
## Step 2: VADER Sentiment Analysis

**What VADER does:** VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon-based sentiment analysis tool. It has a built-in dictionary of ~7,500 words, each pre-scored for sentiment intensity.

**How it works:**
1. Tokenizes the text into words
2. Looks up each word in its lexicon (e.g., "excellent" = +3.2, "terrible" = -3.1)
3. Applies rules for: negation ("not good" flips polarity), intensifiers ("very good" amplifies), capitalization, punctuation
4. Returns four scores:
   - **pos**: proportion of positive words
   - **neg**: proportion of negative words
   - **neu**: proportion of neutral words
   - **compound**: normalized aggregate score from -1 (most negative) to +1 (most positive)

**Why VADER over RoBERTa/BERT:** For ~220 reviews, a dictionary-based tool is sufficient and more interpretable. Transformer models are overkill for this dataset size and harder to explain in a consulting context.

In [ ]:
# Initialize the VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# Let's first see how VADER scores a single review
sample_text = df['text'].iloc[0]
scores = analyzer.polarity_scores(sample_text)

print('=== SINGLE REVIEW EXAMPLE ===')
print(f'Text: {sample_text[:150]}...')
print(f'\nVADER Scores:')
print(f'  Positive: {scores["pos"]:.3f} (proportion of positive words)')
print(f'  Neutral:  {scores["neu"]:.3f} (proportion of neutral words)')
print(f'  Negative: {scores["neg"]:.3f} (proportion of negative words)')
print(f'  Compound: {scores["compound"]:.3f} (overall score, -1 to +1)')

In [ ]:
# Now score every review in the dataset
sentiments = df['text'].apply(lambda x: analyzer.polarity_scores(x))

df['vader_compound'] = sentiments.apply(lambda x: x['compound'])
df['vader_pos'] = sentiments.apply(lambda x: x['pos'])
df['vader_neg'] = sentiments.apply(lambda x: x['neg'])
df['vader_neu'] = sentiments.apply(lambda x: x['neu'])

# Classify into sentiment categories
# Thresholds: compound >= 0.05 = Positive, <= -0.05 = Negative, else Neutral
df['sentiment'] = df['vader_compound'].apply(
    lambda x: 'Positive' if x >= 0.05 else ('Negative' if x <= -0.05 else 'Neutral')
)

print('=== SENTIMENT DISTRIBUTION ===')
print(df['sentiment'].value_counts())
print(f'\nOverall mean compound score: {df["vader_compound"].mean():.3f}')

In [ ]:
# Sentiment breakdown by source
print('=== MEAN SENTIMENT BY SOURCE ===')
source_sentiment = df.groupby('source')['vader_compound'].agg(['mean', 'count', 'std']).round(3)
source_sentiment = source_sentiment.sort_values('mean')
print(source_sentiment)

print('\nKey insight: Reddit has the lowest mean sentiment (most critical/unfiltered).')
print('G2 has the highest (reviews are often incentivized on G2).')

In [ ]:
# Visualize: Sentiment distribution by source
fig = px.histogram(
    df, x='vader_compound', color='source', 
    nbins=30, barmode='overlay', opacity=0.7,
    title='VADER compound score distribution by source',
    labels={'vader_compound': 'Compound Score (-1 = negative, +1 = positive)', 'count': 'Count'},
    color_discrete_sequence=['#1a5276', '#27ae60', '#f39c12', '#e74c3c']
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Visualize: Sentiment pie chart
sent_counts = df['sentiment'].value_counts()
colors = {'Positive': '#27ae60', 'Neutral': '#f39c12', 'Negative': '#e74c3c'}

fig = go.Figure(data=[go.Pie(
    labels=sent_counts.index,
    values=sent_counts.values,
    marker_colors=[colors[s] for s in sent_counts.index],
    hole=0.4,
    textinfo='label+percent'
)])
fig.update_layout(title='Overall sentiment distribution', height=400)
fig.show()

### Step 2b: Validate VADER Against Star Ratings

If VADER is working correctly, its compound scores should correlate with the actual star ratings that users gave. A 5-star review should have a higher compound score than a 2-star review.

We calculate the **Pearson correlation** between star ratings and VADER scores. A positive correlation (closer to +1) means VADER's scoring aligns with how reviewers actually rate the product.

In [ ]:
# Only use reviews that have star ratings (not Reddit)
rated_df = df[df['rating'] > 0].copy()

print(f'Reviews with star ratings: {len(rated_df)}')

# Calculate Pearson correlation
correlation = rated_df[['rating', 'vader_compound']].corr().iloc[0, 1]
print(f'Pearson correlation (rating vs VADER): {correlation:.3f}')

if correlation > 0.3:
    print('Moderate-to-strong positive correlation — VADER scoring aligns with star ratings.')
elif correlation > 0:
    print('Weak positive correlation — VADER captures directional sentiment but with noise.')
else:
    print('No correlation — VADER may not be well-suited for this text type.')

# Scatter plot
fig = px.scatter(
    rated_df, x='rating', y='vader_compound', color='source',
    opacity=0.6, title=f'Star rating vs VADER sentiment (r = {correlation:.3f})',
    labels={'rating': 'Star Rating', 'vader_compound': 'VADER Compound Score'},
    color_discrete_sequence=['#1a5276', '#27ae60', '#f39c12']
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Most positive and most negative reviews
print('=== TOP 5 MOST POSITIVE REVIEWS ===')
for _, row in df.nlargest(5, 'vader_compound').iterrows():
    print(f'[{row["source"]}] Score: {row["vader_compound"]:.2f} | Rating: {row["rating"]}')
    print(f'  {row["text"][:120]}...\n')

print('\n=== TOP 5 MOST NEGATIVE REVIEWS ===')
for _, row in df.nsmallest(5, 'vader_compound').iterrows():
    print(f'[{row["source"]}] Score: {row["vader_compound"]:.2f} | Rating: {row["rating"]}')
    print(f'  {row["text"][:120]}...\n')

---
## Step 3: Pain Point Categorization

**What this does:** Takes every negative review AND every "Cons" section, then checks each one against keyword lists for 8 categories. If the text contains keywords from a category, it gets counted.

**Why keyword-based instead of ML:** For pain point analysis, we want categories that are directly actionable for a PS team. Predefined categories like "Learning Curve" or "Cloud & Migration" are more useful than whatever an unsupervised model would generate. The categories were designed based on a manual review of the data.

**Note:** A single review can match multiple categories. A review saying "the interface is slow and hard to learn" would count under both UI/Usability and Learning Curve.

In [ ]:
# Define pain point categories with keyword lists
pain_categories = {
    'User Interface & Usability': [
        'interface', 'ui', 'clunky', 'unintuitive', 'confusing', 'navigate',
        'navigation', 'cluttered', 'dated', 'modern', 'layout',
        'user friendly', 'cumbersome', 'complicated', 'complex', 'overwhelming',
        'intuitive', 'clumsy', 'awkward', 'frustrating'
    ],
    'Learning Curve': [
        'learning curve', 'steep', 'training', 'learn', 'difficult to learn',
        'onboarding', 'documentation', 'tutorial', 'help', 'manual',
        'figure out', 'getting started', 'hard to understand'
    ],
    'Performance & Speed': [
        'slow', 'lag', 'loading', 'freeze', 'crash', 'hang', 'speed',
        'performance', 'buffer', 'unresponsive', 'timeout', 'sluggish'
    ],
    'Cloud & Migration': [
        'cloud', 'migration', 'migrate', 'desktop', 'transition', 'upgrade',
        'version', 'cloudbridge', 'cloud version', 'desktop version'
    ],
    'Cost & Licensing': [
        'expensive', 'cost', 'price', 'pricing', 'license', 'licensing',
        'subscription', 'fee', 'value', 'money', 'overpriced'
    ],
    'Customer Support': [
        'support', 'customer service', 'help desk', 'ticket', 'response time',
        'technical support'
    ],
    'Features & Functionality': [
        'feature', 'functionality', 'missing', 'lack', 'limited', 'basic',
        'customization', 'customize', 'flexibility', 'integration',
        'export', 'import', 'template', 'formatting'
    ],
    'Updates & Bugs': [
        'bug', 'error', 'glitch', 'issue', 'fix', 'update', 'patch',
        'broken', 'crash', 'not working', 'fails', 'problem'
    ]
}

print(f'Defined {len(pain_categories)} pain point categories')
for cat, keywords in pain_categories.items():
    print(f'  {cat}: {len(keywords)} keywords')

In [ ]:
# Collect all negative text: negative reviews + cons sections
negative_reviews = df[df['sentiment'] == 'Negative']['text'].tolist()
cons_texts = df[df['cons'].str.len() > 5]['cons'].tolist()
all_complaint_texts = negative_reviews + cons_texts

print(f'Negative reviews: {len(negative_reviews)}')
print(f'Cons sections (non-empty): {len(cons_texts)}')
print(f'Total complaint texts to analyze: {len(all_complaint_texts)}')

# Match each complaint against categories
pain_results = {}
for category, keywords in pain_categories.items():
    matches = []
    for text in all_complaint_texts:
        text_lower = text.lower()
        matched_keywords = [kw for kw in keywords if kw in text_lower]
        if matched_keywords:
            matches.append({'text': text, 'matched_keywords': matched_keywords})
    
    pain_results[category] = {
        'count': len(matches),
        'percentage': round(len(matches) / max(len(all_complaint_texts), 1) * 100, 1),
        'samples': matches[:3]
    }

# Sort by count descending
pain_results = dict(sorted(pain_results.items(), key=lambda x: x[1]['count'], reverse=True))

print('\n=== PAIN POINT RANKING ===')
for cat, data in pain_results.items():
    print(f'{cat}: {data["count"]} mentions ({data["percentage"]}%)')

In [ ]:
# Visualize pain points
pain_df = pd.DataFrame([
    {'Category': cat, 'Mentions': data['count']}
    for cat, data in pain_results.items()
    if data['count'] > 0
]).sort_values('Mentions', ascending=True)

fig = go.Figure(go.Bar(
    x=pain_df['Mentions'],
    y=pain_df['Category'],
    orientation='h',
    marker_color='#e74c3c',
    text=pain_df['Mentions'],
    textposition='auto'
))
fig.update_layout(
    title='Pain point frequency in negative reviews and cons',
    xaxis_title='Number of mentions',
    height=400
)
fig.show()

In [ ]:
# Show sample quotes for top 3 pain points
print('=== TOP PAIN POINTS WITH SAMPLE QUOTES ===')
for cat, data in list(pain_results.items())[:3]:
    print(f'\n--- {cat} ({data["count"]} mentions) ---')
    for sample in data['samples'][:2]:
        text_preview = sample['text'][:150]
        keywords = ', '.join(sample['matched_keywords'])
        print(f'  Keywords matched: [{keywords}]')
        print(f'  "{text_preview}..."\n')

---
## Step 4: Topic Modeling (TF-IDF + NMF)

**What this does:** Instead of relying on predefined categories, topic modeling uses unsupervised ML to discover what themes naturally emerge from the review text.

**Two-step process:**

### Step 4a: TF-IDF (Term Frequency-Inverse Document Frequency)
Converts text into numerical vectors. For each word in each review:
- **TF (Term Frequency)**: How often does this word appear in THIS review?
- **IDF (Inverse Document Frequency)**: How rare is this word across ALL reviews?
- **TF-IDF = TF × IDF**: Words that appear frequently in one review but rarely overall get high scores. These are the distinctive words.

Common words like "the", "and", "software" get low scores because they appear everywhere.

### Step 4b: NMF (Non-negative Matrix Factorization)
Takes the TF-IDF matrix (221 reviews × 1000 features) and decomposes it into:
- **W matrix** (221 reviews × 7 topics): How much each review belongs to each topic
- **H matrix** (7 topics × 1000 words): How much each word contributes to each topic

The top words in each topic column of H define what that topic is about.

**Why NMF over LDA:** NMF produces more interpretable, coherent topics for short-text corpora like reviews. LDA assumes a Dirichlet distribution that works better with longer documents.

In [ ]:
# Step 4a: TF-IDF Vectorization

# Custom stop words: remove platform artifacts and generic review language
custom_stops = [
    'caseware', 'software', 'use', 'using', 'used', 'product', 'tool',
    'like', 'really', 'also', 'would', 'could', 'one', 'get', 'got',
    'much', 'make', 'made', 'well', 'good', 'great', 'just', 'know',
    'think', 'want', 'need', 'way', 'thing', 'time', 'year', 'years',
    'lot', 'able', 'work', 'working', 'even', 'still', 'though',
    'overall', 'review', 'recommend', 'rating', 'experience', 'best',
    'worst', 'better', 'however', 'although', 'said', 'say', 'see',
    # Scraping artifacts
    'collected', 'hosted', 'com', 'thumbnail', 'sidebar', 'promoted',
    'post', 'image', 'avatar', 'show', 'details', 'read', 'click',
    'sign', 'join', 'log', 'continue', 'skip', 'content', 'navigation',
    'reply', 'upvote', 'downvote', 'share', 'comments', 'comment',
    'validated', 'incentivized', 'invite', 'emp', 'guest', 'upvotes',
    'reviewer', 'source', 'verified', 'user', 'linkedin', 'profile',
    # Foreign language fragments
    'el', 'em', 'en', 'fr', 'je', 'la', 'las', 'los', 'que', 'se',
    'uso', 'uma', 'da', 'es', 'muy', 'por', 'para', 'mas', 'nos',
    # G2 / platform boilerplate
    'answers', 'real', 'honest', 'pros', 'cons', 'website', 'learn',
    'information', 'comparisons', 'practical', 'recommendations',
    'alternatives', 'g2', 'capterra', 'view', 'explore', 'workflows',
    'discussions', 'currently', 'available', 'visit', 'users', 'fewer',
    # Misc noise
    'gu', 'orb', 'vpa', 'vr', 'bin', 'lol', 'hey', 'ok', 'yes',
    'guy', 'guys', 'fan', 'bit', 'big', 'bad', 'end', 'far',
    'mid', 'non', 'pre', 'set', 'add', 'cut', 'did', 'does',
    'pay', 'job', 'tr', 'ic'
]

# Build TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    max_df=0.85,       # Ignore words appearing in >85% of reviews (too common)
    min_df=2,          # Ignore words appearing in <2 reviews (too rare)
    max_features=1000, # Keep top 1000 features
    stop_words='english',  # Remove standard English stop words
    ngram_range=(1, 2),    # Include single words AND two-word phrases
    token_pattern=r'(?u)\\b[a-zA-Z]{3,}\\b'  # Only alphabetic tokens, 3+ chars
)

# Add custom stops to the built-in English stop words
vectorizer.stop_words = list(vectorizer.stop_words) + custom_stops

# Fit and transform the text data
texts = df['text'].tolist()
tfidf_matrix = vectorizer.fit_transform(texts)
feature_names = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')
print(f'  -> {tfidf_matrix.shape[0]} reviews x {tfidf_matrix.shape[1]} word features')
print(f'\\nSample features: {list(feature_names[:20])}')
print(f'\\nSample bigram features: {[f for f in feature_names if " " in f][:15]}')

In [ ]:
# Show TF-IDF scores for a single review to understand what it produces
sample_idx = 0
sample_tfidf = tfidf_matrix[sample_idx].toarray().flatten()
top_indices = sample_tfidf.argsort()[-10:][::-1]

print(f'=== TF-IDF SCORES FOR REVIEW #{sample_idx} ===')
print(f'Text: {texts[sample_idx][:120]}...\n')
print('Top 10 distinctive words (highest TF-IDF scores):')
for idx in top_indices:
    if sample_tfidf[idx] > 0:
        print(f'  "{feature_names[idx]}": {sample_tfidf[idx]:.4f}')

print('\nThese are the words that make this review distinctive from all other reviews.')

In [ ]:
# Step 4b: NMF Topic Modeling

n_topics = 7  # Number of topics to extract

nmf_model = NMF(
    n_components=n_topics,
    random_state=42,   # For reproducibility
    max_iter=500       # Convergence iterations
)

# W = reviews-to-topics matrix, H = topics-to-words matrix
W = nmf_model.fit_transform(tfidf_matrix)  # Shape: (221 reviews, 7 topics)
H = nmf_model.components_                   # Shape: (7 topics, 1000 words)

print(f'W matrix (reviews → topics): {W.shape}')
print(f'H matrix (topics → words): {H.shape}')

# Extract top words per topic
print(f'\n=== DISCOVERED TOPICS ===')
n_top_words = 8
topics = []
for topic_idx, topic_weights in enumerate(H):
    top_word_indices = topic_weights.argsort()[-n_top_words:][::-1]
    top_words = [feature_names[i] for i in top_word_indices]
    topics.append(top_words)
    print(f'\nTopic {topic_idx + 1}: {" · ".join(top_words)}')

In [ ]:
# Show which topic each review is most strongly associated with
df['dominant_topic'] = W.argmax(axis=1) + 1  # +1 for 1-indexed
df['topic_strength'] = W.max(axis=1)         # How strongly it belongs

print('=== REVIEWS PER TOPIC ===')
print(df['dominant_topic'].value_counts().sort_index())

print('\n=== STRONGEST EXAMPLE FOR EACH TOPIC ===')
for topic_num in range(1, n_topics + 1):
    topic_reviews = df[df['dominant_topic'] == topic_num].nlargest(1, 'topic_strength')
    if len(topic_reviews) > 0:
        row = topic_reviews.iloc[0]
        print(f'\nTopic {topic_num} ({" · ".join(topics[topic_num-1][:4])}):')
        print(f'  [{row["source"]}] {row["text"][:120]}...')

---
## Step 5: Product Strengths

Same keyword approach as pain points, but applied to positive reviews and pros sections.

In [ ]:
strength_categories = {
    'Comprehensive Audit Platform': ['comprehensive', 'complete', 'all in one', 'robust', 'powerful', 'everything'],
    'Document Organization': ['organize', 'organization', 'structured', 'repository', 'document', 'filing'],
    'Financial Reporting': ['financial statement', 'reporting', 'financial report', 'gifi', 'statements'],
    'Collaboration': ['collaborate', 'collaboration', 'team', 'share', 'real time', 'together'],
    'Cloud Accessibility': ['cloud', 'anywhere', 'remote', 'access', 'browser', 'online'],
    'Automation': ['automate', 'automation', 'automatic', 'efficient', 'time saving', 'streamline'],
    'Compliance & Standards': ['compliance', 'standard', 'cas', 'aspe', 'ifrs', 'regulatory', 'methodology'],
    'Trial Balance & Working Papers': ['trial balance', 'working paper', 'lead sheet', 'engagement', 'workpaper']
}

positive_texts = df[df['sentiment'] == 'Positive']['text'].tolist()
pros_texts = df[df['pros'].str.len() > 5]['pros'].tolist()
all_positive = positive_texts + pros_texts

print('=== PRODUCT STRENGTHS ===')
for category, keywords in strength_categories.items():
    matches = sum(1 for text in all_positive if any(kw in text.lower() for kw in keywords))
    if matches > 2:
        print(f'  {category}: {matches} positive mentions')

---
## Step 6: Summary — Implications for Professional Services

This is where the NLP analysis translates into actionable consulting intelligence.

In [ ]:
print('=' * 60)
print('CASEWARE CUSTOMER VOICE — ANALYSIS SUMMARY')
print('=' * 60)

print(f'\nDataset: {len(df)} reviews from {len(df["source"].unique())} sources')
print(f'Sentiment: {len(df[df.sentiment=="Positive"])} positive, {len(df[df.sentiment=="Neutral"])} neutral, {len(df[df.sentiment=="Negative"])} negative')
print(f'Mean sentiment: {df.vader_compound.mean():.3f} (scale: -1 to +1)')

print(f'\n--- TOP 3 PAIN POINTS ---')
for cat, data in list(pain_results.items())[:3]:
    print(f'  1. {cat}: {data["count"]} mentions ({data["percentage"]}%)')

print(f'\n--- PS TEAM IMPLICATIONS ---')
top_pain = list(pain_results.keys())[0]
if 'User Interface' in top_pain:
    print('  → UI concerns are #1. Spend extra time during training on workflow shortcuts.')
if pain_results.get('Learning Curve', {}).get('count', 0) > 10:
    print('  → Learning curve is significant. Budget extra training time in SOW.')
if pain_results.get('Cloud & Migration', {}).get('count', 0) > 5:
    print('  → Cloud migration generates mixed sentiment. Structured change management needed.')

reddit_df = df[df['source'] == 'Reddit']
if len(reddit_df) > 0:
    reddit_mean = reddit_df['vader_compound'].mean()
    g2_mean = df[df['source'] == 'G2']['vader_compound'].mean()
    print(f'  → Reddit sentiment ({reddit_mean:.2f}) vs G2 ({g2_mean:.2f}): community forums are harsher.')
    print('    New users may arrive with negative preconceptions. Proactive onboarding helps.')

print(f'\n--- KEY INSIGHT ---')
print('The gap between G2 sentiment (0.82) and Reddit sentiment (0.40) suggests')
print('that verified review platforms capture a more positive view than the unfiltered')
print('practitioner experience. PS consultants should be prepared for the Reddit-level')
print('sentiment when working with hands-on users during implementation.')